In [0]:
from pyspark.sql.functions import count, avg, sum, month


In [0]:
# Load rental table
rental_df = spark.read.csv("/Workspace/Users/partho.21partho@gmail.com/DVD_data/rental.csv", header=True, inferSchema=True)
rental_df.show(5)

# Load payment table
payment_df = spark.read.csv("/Workspace/Users/partho.21partho@gmail.com/DVD_data/payment.csv", header=True, inferSchema=True)
payment_df.show(5)



+---------+-------------------+------------+-----------+-------------------+--------+-------------------+
|rental_id|        rental_date|inventory_id|customer_id|        return_date|staff_id|        last_update|
+---------+-------------------+------------+-----------+-------------------+--------+-------------------+
|        2|2005-05-24 22:54:33|        1525|        459|2005-05-28 19:40:33|       1|2006-02-16 02:30:53|
|        3|2005-05-24 23:03:39|        1711|        408|2005-06-01 22:12:39|       1|2006-02-16 02:30:53|
|        4|2005-05-24 23:04:41|        2452|        333|2005-06-03 01:43:41|       2|2006-02-16 02:30:53|
|        5|2005-05-24 23:05:21|        2079|        222|2005-06-02 04:33:21|       1|2006-02-16 02:30:53|
|        6|2005-05-24 23:08:07|        2792|        549|2005-05-27 01:32:07|       1|2006-02-16 02:30:53|
+---------+-------------------+------------+-----------+-------------------+--------+-------------------+
only showing top 5 rows
+----------+----------

In [0]:

# 1. Get total number of payments

total_payments = payment_df.count()
print("Total number of payment : ",total_payments)

Total number of payment :  14596


In [0]:
# 2. Get average payment amount
avg_payment = payment_df.select(avg("amount").alias("average_amount"))
avg_payment.show()

+----------------+
|  average_amount|
+----------------+
|4.20060564538219|
+----------------+



In [0]:
# 3. Aggregate payments by staff and find the best staff (highest total)

payments_by_staff = (
    payment_df
    .groupBy("staff_id")
    .agg(sum("amount").alias("total_collected"))
    .orderBy("total_collected")
)

payments_by_staff.show()


# Find the best staff (highest total amount collected)
best_staff_id = (
    payment_df
    .groupBy("staff_id")
    .agg(sum("amount").alias("total_amount"))
    .orderBy("total_amount", ascending=False)
    .limit(1)
)
print("Best Staff ID : ",best_staff_id.collect()[0][0])
best_staff_id.show()

+--------+------------------+
|staff_id|   total_collected|
+--------+------------------+
|       1| 30252.12000000458|
|       2|31059.920000004768|
+--------+------------------+

Best Staff ID :  2
+--------+------------------+
|staff_id|      total_amount|
+--------+------------------+
|       2|31059.920000004768|
+--------+------------------+



In [0]:
# 4. Count rentals per month

rentals_per_month = (
    rental_df
    .withColumn("rental_month", month("rental_date"))
    .groupBy("rental_month")
    .count()
    .orderBy("rental_month")
)

rentals_per_month.show()

+------------+-----+
|rental_month|count|
+------------+-----+
|           2|  182|
|           5| 1156|
|           6| 2311|
|           7| 6709|
|           8| 5686|
+------------+-----+



In [0]:
# 5. Total payments collected by each staff(join rental and payment tables)


# Join payment and rental tables
payment_rental_join = (
    payment_df
    .join(rental_df, payment_df.rental_id == rental_df.rental_id)
)

# Aggregate total payments collected by each staff member
total_payment_by_staff = (
    payment_rental_join
    .groupBy(rental_df.staff_id)
    .agg(sum(payment_df.amount).alias("total_collected"))
)

total_payment_by_staff.show()



+--------+------------------+
|staff_id|   total_collected|
+--------+------------------+
|       1|30498.710000002757|
|       2| 30813.33000000281|
+--------+------------------+

